In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import ast

In [ ]:
credits = pd.read_csv("tmdb_5000_credits.csv")
credits.head()
credits.shape

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
movies.head()
movies.shape
movies.isna().sum()

Merge both the datset based on title 

In [ ]:
movies.drop_duplicates()
# movies = movies.merge(credits,on='title')

movies.head(2)
movies.shape

remove un important columns 
# budget - no
# geners
# id 
# keywords
# original language - No
# original Title - No
# title - yes 
# overview - Yes 
# popularity - NO
# production company - NO
# relese period - NO
# revenue - no
# cast - yes 
# crew - yes

In [ ]:

# movies = movies.drop(columns=['movie_id_x', 'cast_x', 'crew_x',
#        'movie_id_y', 'cast_y', 'crew_y'])
print(movies.columns)


In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast', 'crew']]
movies.head()

In [ ]:
movies.isnull().sum()


In [ ]:
movies.dropna(inplace=True)
movies.shape


In [ ]:
movies.iloc[0].genres

genres present in very wiered form i have to convert this dictionary into list 


In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L


In [ ]:
movies['genres'] =  movies['genres'].apply(convert)
movies.head()

same for other features 


In [ ]:
movies['keywords'] =  movies['keywords'].apply(convert)
movies.head


In [ ]:
# movies.iloc[0].cast

def convert_cast(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3 :
            L.append(i['name'])
            counter += 1
        else:
            break
    return L


In [ ]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [ ]:
movies.head()

In [ ]:
movies['crew'][0]

In [ ]:
def fatch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director' :
            L.append(i['name'])
            break;

    return L



In [ ]:
movies['crew'] = movies['crew'].apply(fatch_director)

In [ ]:
movies.head()

In [ ]:
type(movies['overview'][0])


overview is string so i have to convert it to the list

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x : x.split())
movies.head()

now remove space between all the words 

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x ])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x ])

movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x ])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x ])
movies.head()

Create new column tags in whic we will concatinate all the columns 

In [ ]:
movies['tags'] = movies['overview'] + movies['genres']+movies['keywords']+movies['cast']+movies['crew']
movies.head()

In [ ]:
movies.columns

In [ ]:
new_df = movies[['movie_id','title','tags']]
new_df.head()

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x) if isinstance(x, list) else x)

In [ ]:
new_df.head()

In [ ]:
new_df['tags'][0]
new_df['tags'] = new_df['tags'].apply(lambda x : x.lower())


In [ ]:
new_df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features= 5000,stop_words='english')

convert to array because most of the value are 0 because ek tag mai 5000 values  to ho nhi sakti 

🔹 fit_transform()

👉 2 kaam ek saath karta hai:

fit() → words learn karta hai (vocabulary banata hai)
transform() → text ko numbers (vectors) me convert karta hai

🔹 Output kya hota hai?

👉 Sparse matrix (memory efficient format)

👉 Sparse matrix ko normal array (numbers table) me convert karta hai
👉 Easy for:

understanding
cosine similarity

In [ ]:
vector = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vector

In [ ]:
cv.get_stop_words()

In [ ]:
for i in cv.get_feature_names_out():
    print(i)

award
awards 
this is done but it is a problem award and awards are same 
apply steaming to solve this problem 


In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))
    
    return " ".join(y)

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
stem("lovely")
stem("love")
stem("dancing")
stem("singing")
stem("actors")
stem("actresses")

In [ ]:

print(new_df['tags'][0])

In [ ]:
print(new_df['tags'][2])


i have total 4806 movie vector - and har vector mai 5000 words h 
# 5000 dimension space coordinate h ab hame hr movie ka har movie sai sath distance calculate karna h euclidean nhi cosine distance 

Distance = 1 / similirity 
# distance kam  similirity jada , distance jada similarity kam 


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(vector).shape

In [ ]:
similarity = cosine_similarity(vector)

In [ ]:
similarity[1]

In [ ]:
new_df.head()

In [ ]:
new_df[new_df['title'] == 'Nixon'].index[0]

In [ ]:
def recommended(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)),reverse=True,key= lambda x: x[1])[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)


In [ ]:
new_df['title']

In [ ]:
recommended('The American President')

In [ ]:
print((new_df['title'] == 'Cinderella').sum())

In [ ]:
print(new_df[new_df['title'] == 'Cinderella'])

use pickle library to import your model 
# WB menas write binary use when we have to write binary data into file 

In [ ]:
import pickle
pickle.dump(new_df.to_dict(),open('movies_dict.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))


In [ ]:
new_df.to_dict()

In [ ]:
new_df.head()

In [ ]:
print(movies.head())

In [ ]:
print(new_df.columns)